In [ ]:
# Written by Milo

In [ ]:
!curl -o 'topics.txt' 'https://api.manifold.markets/v0/groups'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  327k  100  327k    0     0  1639k      0 --:--:-- --:--:-- --:--:-- 1644k


In [ ]:
import json
btopics = json.load(open('topics.txt', 'r', encoding='utf-8'))
stopics = sorted(btopics, key= lambda x: -x['totalMembers'] )

In [ ]:
names = [ (i, topic['name'], topic['id'], topic['totalMembers']) for i, topic in enumerate(stopics[:100])]
topics = names[0], names[1], names[2], names[3], names[4], names[5], names[6], names[7], names[11], names[13], names[16], names[17], names[18], names[19], names[22], names[26], names[29], names[35], names[41], names[45], names[48], names[53], names[54], names[59], names[62]
ids = [ topic[2] for topic in topics ]

In [ ]:
import requests

In [ ]:
session = requests.Session()

In [ ]:
rmarkets = []
for i in ids:
    params = { 'groupId': i, 'limit': 1000 }
    rmarkets += session.get('https://api.manifold.markets/v0/markets', params=params).json()

In [ ]:
mconditions = lambda market : (
    market.get('isResolved') and
    market.get('outcomeType') == 'BINARY' and
    market.get('resolution') != 'CANCEL' and
    market.get('createdTime') >= 1748736000000 and
    market.get('uniqueBettorCount') >= 30 # Trying to proxy that the markets are about something a bit real, not just fun
)

idtom = { market['id']: market for market in rmarkets if mconditions(market) }
markets = list(idtom.values())

In [ ]:
rbets = []
day = 24 * 60 ** 2 * 1000 # A day
for market in markets:
    params = { 'contractId': market['id'], 'beforeTime': market['closeTime'] - day }
    mbets = session.get('https://api.manifold.markets/v0/bets', params=params).json()
    rbets += mbets

In [ ]:
bconditions = lambda bet: (
    idtom[bet['contractId']]['creatorId'] != bet['userId'] and
    bet['amount'] > 0
)

bets = [ bet for bet in rbets if bconditions(bet)]

In [ ]:
utohis = {}
userc = 0
for bet in bets:
    user_id = bet['userId']
    if user_id not in utohis.keys():
        params = { 'userId': user_id, 'period': 'allTime' }
        history = session.get('https://api.manifold.markets/v0/get-user-portfolio-history', params=params).json()
        utohis[user_id] = history
        userc += 1

In [ ]:
import math

In [ ]:
def balance_prop(bet):
    user_id = bet['userId']
    history = utohis[user_id]
    btime = bet['createdTime']
    for i in range(1, len(history)):
        if history[i]['timestamp'] > btime:
            try:
                return (bet['amount'] + sum(bet['fees'].values())) / history[i-1]['balance']
            except:
                continue
    return 0

In [ ]:
def rkelly(pb, bprop):
    return bprop * (1 - pb) + pb

In [ ]:
# In a NO bet, you lower the probability of the respective market
# You're betting in the NOT(Event) market
# The probBefore in that market is the inverse of the probBefore of the positive market, 1 - probBefore
# Everything else is otherwise identical
def prob_before(bet):
    return bet['probBefore'] if bet['outcome'] == 'YES' else 1 - bet['probBefore']

# And the kelly prob you get out is your degree of belief in the negative market
# 1 - your negative kelly prob is your positive kelly prob
def dir_kelly(bet):
    pb = prob_before(bet)
    bprop = balance_prop(bet)
    return rkelly(pb, bprop)

# For nll:
# If the outcome of the market is YES, your positive kelly prob is your prob
# Otherwise, the market outcome is NO, so 1 - positive kelly prob is your prob
# Your directional kelly prob will always be for the market your bet concerns
# So, either your outcome is equal to the market outcome,
#     in which case whatever your directed kelly prob is is the prob of the resolution which happened
# Or, it's not
#     in which case, the inverse is
def outcome_prob(market, bet):
    dkelly = dir_kelly(bet)
    return dkelly if market['resolution'] == bet['outcome'] else 1 - dkelly

In [ ]:
utops = {}
for bet in bets:
    user_id = bet['userId']
    market = idtom[bet['contractId']]
    prob = outcome_prob(market, bet)
    utops.setdefault(user_id, []).append(prob)

In [ ]:
utops = { key: val for key, val in utops.items() if len(val) > 1 }

In [ ]:
import numpy

In [ ]:
from math import sqrt

In [ ]:
# Clipping, but a LITTLE sus that I'm needing to, double check equations
# But I'm thinking it's just that the balance fractions are a bit off
eps = 1e-6
utops = {
    uid: numpy.clip(numpy.array(probs), eps, 1 - eps)
    for uid, probs in utops.items()
}

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from scipy import stats

In [ ]:
def probs_lower(probs):
  mean = probs.mean()
  ci = stats.t.interval(
      confidence=0.95,
      df=len(probs)-1,
      loc=mean,
      scale=stats.sem(probs)
  )
  return ci[0]

In [ ]:
# I made an effort, but it is just very hard to be more principled than just filtering people by some arbitrary number of trades
# In order to get rid of people who just got lucky
umeans = [ [ uid, probs_lower(probs) ] for uid, probs in utops.items() if len(probs) > 1 ]
sorted_means = sorted(umeans, key=lambda u: -u[1])

/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2323: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2324: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


In [ ]:
# Get highest performing users in plat or better
top_users = []
for uid_mean in sorted_means:
    uid = uid_mean[0]
    leagues = session.get(f'https://api.manifold.markets/v0/leagues?userId={uid}').json()
    if not leagues:
        continue
    rec_league = max(leagues, key=lambda league: league.get('season', 0))
    rec_div = rec_league.get('division')

    if rec_div is not None and rec_div >= 4:
        user = session.get(f'https://api.manifold.markets/v0/user/by-id/{uid}').json()
        top_users.append([user, uid_mean[1]])
        if len(top_users) >= 30:
            break

In [ ]:
[ (i, unllpair[0]['url'], unllpair[1]) for i, unllpair in enumerate(top_users[:100]) ]

[(0, 'https://manifold.markets/fornever', np.float64(0.8535540300311962)),
 (1, 'https://manifold.markets/PogoStick', np.float64(0.8299867012069694)),
 (2, 'https://manifold.markets/MugaSofer', np.float64(0.8200755629084194)),
 (3, 'https://manifold.markets/HillaryClinton', np.float64(0.814882816101161)),
 (4, 'https://manifold.markets/Mana', np.float64(0.8010428248969014)),
 (5, 'https://manifold.markets/Luxeed', np.float64(0.7895698482487844)),
 (6, 'https://manifold.markets/Valchap', np.float64(0.7799467125355053)),
 (7,
  'https://manifold.markets/FlorisvanDoorn',
  np.float64(0.7440553522569834)),
 (8, 'https://manifold.markets/Arky', np.float64(0.7432129756274546)),
 (9,
  'https://manifold.markets/ChristopherRandles',
  np.float64(0.7361615283812064)),
 (10,
  'https://manifold.markets/NiklasBergstrom',
  np.float64(0.7337120451587348)),
 (11, 'https://manifold.markets/Chumchulum', np.float64(0.7266186642147381)),
 (12, 'https://manifold.markets/Calibrate', np.float64(0.72236855